# Przygotowanie panelu danych

Notebook przygotowuje panel danych wykorzystany w dalszej analizie
zależności między globalizacją a poziomem i wzrostem gospodarczym
w wybranych państwach Europy Środkowo-Wschodniej.

Procedura obejmuje:

- wczytanie i ujednolicenie danych źródłowych,
- połączenie danych makroekonomicznych z indeksami KOF,
- konstrukcję zmiennych wykorzystywanych w modelach,
- kontrolę kompletności i struktury panelu,
- analizę dostępności danych,
- przygotowanie finalnej próby obejmującej lata 1995–2023.

Finalny zbiór danych jest eksportowany do katalogu `output`
i stanowi dane wejściowe dla notebooka z analizą empiryczną.

## 1. Konfiguracja

In [1]:
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import HTML, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

In [2]:
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "gdp_const": "GDP_constant_2015_USD_transformed.xlsx",
    "labor_force": "total_labor_force_transformed.xlsx",
    "gcf_const": "GrossCapitalFormation_transformed.xlsx",
    "inflation_cpi": "Inflation_consumerpricesannual_transformed.xlsx",
    "fdi_gdp": "foreigndirectinvestmentnetflows_of_GDP_transformed.xlsx",
    "gov_consumption_const": "general_government_final_consumption_expenditure_transformed.xlsx",
    "broad_money_gdp": "broadmoney_transformed.xlsx",
    "education_years_adults": "average_years_schooling_adults_transformed.xlsx",
    "kof": "KOFGI_2025_public.xlsx"
}

CEE = ["BGR", "CZE", "EST", "HUN", "LVA", "LTU", "POL", "ROU", "SVK", "SVN"]

COUNTRY_NAMES = {
    "BGR": "Bulgaria",
    "CZE": "Czechia",
    "EST": "Estonia",
    "HUN": "Hungary",
    "LVA": "Latvia",
    "LTU": "Lithuania",
    "POL": "Poland",
    "ROU": "Romania",
    "SVK": "Slovakia",
    "SVN": "Slovenia",
}

ANALYSIS_START_YEAR = 1990
ANALYSIS_END_YEAR = 2023

### 1.1. Funkcje pomocnicze

Poniższe funkcje odpowiadają za ujednolicenie struktury danych
źródłowych oraz kontrolę unikalności obserwacji państwo–rok.

In [3]:
def load_long_series(path: Path, value_name: str) -> pd.DataFrame:
    df = pd.read_excel(path)

    required = {"iso3", "country", "year", "value"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{path.name}: brakuje kolumn {sorted(missing)}"
        )

    df = df.copy()

    df["iso3"] = (
        df["iso3"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    df["year"] = (
        pd.to_numeric(df["year"], errors="coerce")
        .astype("Int64")
    )

    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce",
    )

    df = df.dropna(subset=["iso3", "year"])

    duplicates = df.duplicated(
        ["iso3", "year"],
        keep=False,
    )

    if duplicates.any():
        preview = (
            df.loc[
                duplicates,
                ["iso3", "country", "year", "value"],
            ]
            .sort_values(["iso3", "year", "country"])
            .head(20)
        )

        raise ValueError(
            f"{path.name}: wykryto duplikaty po iso3–year.\n"
            f"{preview.to_string(index=False)}"
        )

    return (
        df[["iso3", "year", "value"]]
        .rename(columns={"value": value_name})
        .sort_values(["iso3", "year"])
        .reset_index(drop=True)
    )

In [4]:
def assert_unique_country_year(
    df: pd.DataFrame,
    frame_name: str,
) -> None:
    required_columns = {"iso3", "year"}

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{frame_name}: brakuje kolumn "
            f"{sorted(missing_columns)}"
        )

    duplicate_mask = df.duplicated(
        subset=["iso3", "year"],
        keep=False,
    )

    if duplicate_mask.any():
        duplicate_rows = (
            df.loc[duplicate_mask]
            .sort_values(["iso3", "year"])
        )

        display(duplicate_rows.head(30))

        raise ValueError(
            f"{frame_name}: klucz iso3–year "
            "nie jest unikalny."
        )

    print(
        f"{frame_name}: klucz iso3–year jest unikalny "
        f"({len(df)} wierszy)."
    )

In [ ]:
def show_table(
    dataframe: pd.DataFrame,
    index: bool = False,
) -> None:
    display(
        HTML(
            dataframe.to_html(
                index=index,
                border=0,
            )
        )
    )

## 2. Wczytanie i przygotowanie danych źródłowych

W tej części wczytywane są dane makroekonomiczne oraz indeksy globalizacji KOF. Dane są następnie ujednolicane do struktury państwo–rok i przygotowywane do połączenia w jeden panel.

In [6]:
series_frames = {
    "gdp_const": load_long_series(
        DATA_DIR / FILES["gdp_const"],
        "gdp_const",
    ),
    "labor_force": load_long_series(
        DATA_DIR / FILES["labor_force"],
        "labor_force",
    ),
    "gcf_const": load_long_series(
        DATA_DIR / FILES["gcf_const"],
        "gcf_const",
    ),
    "inflation_cpi": load_long_series(
        DATA_DIR / FILES["inflation_cpi"],
        "inflation_cpi",
    ),
    "fdi_gdp": load_long_series(
        DATA_DIR / FILES["fdi_gdp"],
        "fdi_gdp",
    ),
    "gov_consumption_const": load_long_series(
        DATA_DIR / FILES["gov_consumption_const"],
        "gov_consumption_const",
    ),
    "broad_money_gdp": load_long_series(
        DATA_DIR / FILES["broad_money_gdp"],
        "broad_money_gdp",
    ),
    "education_years_adults": load_long_series(
        DATA_DIR / FILES["education_years_adults"],
        "education_years_adults",
    ),
}

for variable_name, frame in series_frames.items():
    assert_unique_country_year(
        frame,
        variable_name,
    )

gdp_const: klucz iso3–year jest unikalny (17556 wierszy).
labor_force: klucz iso3–year jest unikalny (17556 wierszy).
gcf_const: klucz iso3–year jest unikalny (17556 wierszy).
inflation_cpi: klucz iso3–year jest unikalny (17556 wierszy).
fdi_gdp: klucz iso3–year jest unikalny (17556 wierszy).
gov_consumption_const: klucz iso3–year jest unikalny (17556 wierszy).
broad_money_gdp: klucz iso3–year jest unikalny (17556 wierszy).
education_years_adults: klucz iso3–year jest unikalny (6420 wierszy).


### 2.1. Indeksy globalizacji KOF


In [7]:
kof_path = DATA_DIR / FILES["kof"]

kof = pd.read_excel(kof_path)

kof_identifier_map = {
    "ISO3": "iso3",
    "ISO": "iso3",
    "Code": "iso3",
    "country_code": "iso3",
    "Country": "country",
    "COUNTRY": "country",
    "Year": "year",
    "YEAR": "year",
}

kof = kof.rename(columns=kof_identifier_map)

required_kof_identifiers = {"iso3", "year"}
missing_identifiers = required_kof_identifiers - set(kof.columns)

if missing_identifiers:
    raise ValueError(
        "W danych KOF brakuje kolumn identyfikacyjnych: "
        f"{sorted(missing_identifiers)}.\n"
        f"Dostępne kolumny: {kof.columns.tolist()}"
    )

kof["iso3"] = (
    kof["iso3"]
    .astype("string")
    .str.strip()
    .str.upper()
)

kof["year"] = (
    pd.to_numeric(kof["year"], errors="coerce")
    .astype("Int64")
)

kof = kof.dropna(subset=["iso3", "year"])

print("Wymiary KOF po oczyszczeniu:", kof.shape)
print(
    "Zakres lat:",
    kof["year"].min(),
    "-",
    kof["year"].max(),
)
print("Liczba państw:", kof["iso3"].nunique())

Wymiary KOF po oczyszczeniu: (11610, 30)
Zakres lat: 1970 - 2023
Liczba państw: 215


In [8]:
kof_rename_map = {
    # Globalizacja ogółem
    "KOFGI": "kof_overall",
    "KOFGIdf": "kof_overall_de_facto",
    "KOFGIdj": "kof_overall_de_jure",

    # Globalizacja ekonomiczna
    "KOFEcGI": "kof_economic",
    "KOFEcGIdf": "kof_economic_de_facto",
    "KOFEcGIdj": "kof_economic_de_jure",

    # Globalizacja handlowa
    "KOFTrGI": "kof_trade",
    "KOFTrGIdf": "kof_trade_de_facto",
    "KOFTrGIdj": "kof_trade_de_jure",

    # Globalizacja finansowa
    "KOFFiGI": "kof_financial",
    "KOFFiGIdf": "kof_financial_de_facto",
    "KOFFiGIdj": "kof_financial_de_jure",

    # Globalizacja społeczna
    "KOFSoGI": "kof_social",
    "KOFSoGIdf": "kof_social_de_facto",
    "KOFSoGIdj": "kof_social_de_jure",

    # Globalizacja interpersonalna
    "KOFIpGI": "kof_interpersonal",
    "KOFIpGIdf": "kof_interpersonal_de_facto",
    "KOFIpGIdj": "kof_interpersonal_de_jure",

    # Globalizacja informacyjna
    "KOFInGI": "kof_informational",
    "KOFInGIdf": "kof_informational_de_facto",
    "KOFInGIdj": "kof_informational_de_jure",

    # Globalizacja kulturowa
    "KOFCuGI": "kof_cultural",
    "KOFCuGIdf": "kof_cultural_de_facto",
    "KOFCuGIdj": "kof_cultural_de_jure",

    # Globalizacja polityczna
    "KOFPoGI": "kof_political",
    "KOFPoGIdf": "kof_political_de_facto",
    "KOFPoGIdj": "kof_political_de_jure",
}

missing_raw_kof_columns = [
    raw_name
    for raw_name in kof_rename_map
    if raw_name not in kof.columns
]

if missing_raw_kof_columns:
    raise ValueError(
        "W pliku KOF brakuje następujących indeksów: "
        f"{missing_raw_kof_columns}"
    )

kof = kof.rename(columns=kof_rename_map)

In [ ]:
kof_columns = [
    col
    for col in kof.columns
    if col.startswith("kof_")
]

kof = (
    kof[["iso3", "year"] + kof_columns]
    .copy()
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)

for col in kof_columns:
    kof[col] = pd.to_numeric(
        kof[col],
        errors="coerce",
    )

assert_unique_country_year(
    kof,
    "KOF",
)

print("Wymiary ramki KOF do scalania:", kof.shape)
print("Liczba indeksów KOF:", len(kof_columns))

KOF: klucz iso3–year jest unikalny (11610 wierszy).
Wymiary ramki KOF do scalania: (11610, 29)
Liczba indeksów KOF: 27


iso3,year,kof_overall,kof_overall_de_facto,kof_overall_de_jure,kof_economic,kof_economic_de_facto,kof_economic_de_jure,kof_trade,kof_trade_de_facto,kof_trade_de_jure,kof_financial,kof_financial_de_facto,kof_financial_de_jure,kof_social,kof_social_de_facto,kof_social_de_jure,kof_interpersonal,kof_interpersonal_de_facto,kof_interpersonal_de_jure,kof_informational,kof_informational_de_facto,kof_informational_de_jure,kof_cultural,kof_cultural_de_facto,kof_cultural_de_jure,kof_political,kof_political_de_facto,kof_political_de_jure
ABW,1970,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ABW,1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ABW,1972,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ABW,1973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ABW,1974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
kof_cee_preview = (
    kof.loc[
        kof["iso3"].isin(CEE)
        & kof["year"].between(1995, 2023)
    ]
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)

show_table(
    kof_cee_preview.head(10)
)

iso3,year,kof_overall,kof_overall_de_facto,kof_overall_de_jure,kof_economic,kof_economic_de_facto,kof_economic_de_jure,kof_trade,kof_trade_de_facto,kof_trade_de_jure,kof_financial,kof_financial_de_facto,kof_financial_de_jure,kof_social,kof_social_de_facto,kof_social_de_jure,kof_interpersonal,kof_interpersonal_de_facto,kof_interpersonal_de_jure,kof_informational,kof_informational_de_facto,kof_informational_de_jure,kof_cultural,kof_cultural_de_facto,kof_cultural_de_jure,kof_political,kof_political_de_facto,kof_political_de_jure
BGR,1995,58.846550,58.283939,59.409138,50.299397,49.771828,50.826962,54.307018,57.930470,50.683567,46.291771,41.613178,50.970367,50.382584,47.127129,53.638035,45.851303,41.848015,49.854591,45.710201,52.029633,39.390770,59.586239,47.503735,71.668739,75.857651,77.952881,73.762444
BGR,1996,62.689419,62.661381,62.717434,58.266251,60.909443,55.623043,60.084270,72.724899,47.443638,56.448223,49.093994,63.802456,50.920769,47.483482,54.358059,48.513626,42.506531,54.520718,48.335396,52.687447,43.983341,55.913319,47.256489,64.570145,78.881203,79.591194,78.171204
BGR,1997,64.336311,64.632072,64.040550,61.615921,66.241692,56.990162,62.873596,76.164894,49.582291,60.358257,56.318493,64.398026,51.438580,47.553047,55.324108,48.928810,42.318523,55.539093,50.441868,54.045555,46.838184,54.945038,46.295048,63.595032,79.954430,80.101456,79.807388
BGR,1998,64.573990,63.389984,65.757988,60.311325,61.428997,59.193665,61.682072,70.083069,53.281075,58.940586,52.774925,65.106247,52.244350,48.195045,56.293644,49.889969,43.768780,56.011166,51.709763,53.839901,49.579632,55.133293,46.976444,63.290131,81.166298,80.545937,81.786652
BGR,1999,66.648697,64.385185,68.912209,62.156857,61.466988,62.846718,64.659943,69.104698,60.215183,59.653763,53.829273,65.478264,54.678539,51.075325,58.281761,50.947475,44.696651,57.198299,57.592266,60.828991,54.355541,55.495892,47.700344,63.291439,83.110695,80.613235,85.608154
BGR,2000,68.530228,67.376038,69.684395,65.841057,67.670601,64.011513,69.042267,75.596611,62.487949,62.639835,59.744587,65.535080,56.108768,52.668816,59.548710,49.752644,45.916885,53.588402,62.659801,64.178825,61.140781,55.913841,47.910736,63.916946,83.640839,81.788727,85.492950
BGR,2001,68.755013,68.886681,68.623344,63.678036,69.158531,58.197552,68.940598,74.809631,63.071560,58.415478,63.507412,53.323540,58.273922,55.462719,61.085121,51.940498,48.297474,55.583519,64.869713,66.676964,63.062462,58.011555,51.413708,64.609406,84.313103,82.038826,86.587379
BGR,2002,67.657661,68.053459,67.261871,58.138645,67.084824,49.192467,67.391350,71.486954,63.295753,48.885941,62.682693,35.089191,60.773857,56.148796,65.398933,52.929276,48.983105,56.875443,66.677475,66.159843,67.195107,62.714832,53.303429,72.126244,84.060486,80.926750,87.194229
BGR,2003,70.213982,69.368851,71.059120,63.553825,68.505943,58.601707,68.667854,73.253845,64.081848,58.439808,63.758041,53.121571,62.064785,57.839947,66.289619,54.299686,50.466694,58.132679,67.615303,67.290413,67.940186,64.279366,55.762741,72.795998,85.023361,81.760666,88.286034
BGR,2004,70.809647,69.421700,72.197609,66.399002,72.273254,60.524765,72.892021,77.238503,68.545532,59.905991,67.307983,52.503998,61.850903,55.120438,68.581360,56.941631,51.833054,62.050201,64.461548,59.015850,69.907249,64.149544,54.512440,73.786644,84.179047,80.871407,87.486710


### 2.2. Scalanie źródeł i konstrukcja panelu

Dane makroekonomiczne oraz indeksy KOF są łączone według kodu państwa i roku. Scalanie typu outer pozwala zachować informacje o dostępności poszczególnych zmiennych przed późniejszym wyborem próby badawczej.

In [11]:
frames_to_merge = list(series_frames.values()) + [kof]

In [12]:
panel_raw = reduce(
    lambda left, right: left.merge(
        right,
        on=["iso3", "year"],
        how="outer",
        validate="one_to_one",
    ),
    frames_to_merge,
)

panel_raw = (
    panel_raw
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)

assert_unique_country_year(
    panel_raw,
    "panel_raw",
)

print("Wymiary panel_raw:", panel_raw.shape)
print(
    "Liczba państw:",
    panel_raw["iso3"].nunique(),
)
print(
    "Zakres lat:",
    panel_raw["year"].min(),
    "-",
    panel_raw["year"].max(),
)

panel_raw: klucz iso3–year jest unikalny (18111 wierszy).
Wymiary panel_raw: (18111, 37)
Liczba państw: 282
Zakres lat: 1960 - 2025


### 2.3. Konstrukcja pełnej siatki państwo–rok

Dla dziesięciu analizowanych państw tworzona jest pełna siatka obserwacji państwo–rok. Pozwala to zachować lata, dla których część zmiennych jest niedostępna, oraz jawnie zidentyfikować braki danych.

In [ ]:
analysis_grid = pd.MultiIndex.from_product(
    [
        CEE,
        range(
            ANALYSIS_START_YEAR,
            ANALYSIS_END_YEAR + 1,
        ),
    ],
    names=["iso3", "year"],
).to_frame(index=False)

print("Wymiary pełnej siatki państwo–rok:", analysis_grid.shape)

Wymiary pełnej siatki: (340, 2)


In [14]:
panel_raw["year"] = (
    pd.to_numeric(
        panel_raw["year"],
        errors="raise",
    )
    .astype(int)
)

panel_cee_source = (
    panel_raw.loc[
        panel_raw["iso3"].isin(CEE)
        & panel_raw["year"].between(
            ANALYSIS_START_YEAR,
            ANALYSIS_END_YEAR,
        )
    ]
    .copy()
)

assert_unique_country_year(
    panel_cee_source,
    "panel_cee_source",
)

panel_cee_final = analysis_grid.merge(
    panel_cee_source,
    on=["iso3", "year"],
    how="left",
    validate="one_to_one",
)

# Jedna kanoniczna nazwa każdego państwa
panel_cee_final.insert(
    1,
    "country",
    panel_cee_final["iso3"].map(
        COUNTRY_NAMES
    ),
)

panel_cee_final = (
    panel_cee_final
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)

assert_unique_country_year(
    panel_cee_final,
    "panel_cee_final",
)

print("\nPANEL CEE PRZED WYBOREM PRÓBY ESTYMACYJNEJ")

print("Liczba obserwacji:", len(panel_cee_final))
print(
    "Liczba państw:",
    panel_cee_final["iso3"].nunique(),
)
print(
    "Zakres lat:",
    panel_cee_final["year"].min(),
    "-",
    panel_cee_final["year"].max(),
)

panel_cee_source: klucz iso3–year jest unikalny (340 wierszy).
panel_cee_final: klucz iso3–year jest unikalny (340 wierszy).

PANEL CEE PRZED WYBOREM PRÓBY ESTYMACYJNEJ
Liczba obserwacji: 340
Liczba państw: 10
Zakres lat: 1990 - 2023


In [15]:
country_year_check = (
    panel_cee_final
    .groupby(["iso3", "country"])
    .agg(
        liczba_lat=("year", "nunique"),
        pierwszy_rok=("year", "min"),
        ostatni_rok=("year", "max"),
    )
    .reset_index()
)

show_table(country_year_check)

iso3,country,liczba_lat,pierwszy_rok,ostatni_rok
BGR,Bulgaria,34,1990,2023
CZE,Czechia,34,1990,2023
EST,Estonia,34,1990,2023
HUN,Hungary,34,1990,2023
LTU,Lithuania,34,1990,2023
LVA,Latvia,34,1990,2023
POL,Poland,34,1990,2023
ROU,Romania,34,1990,2023
SVK,Slovakia,34,1990,2023
SVN,Slovenia,34,1990,2023


In [16]:
value_columns = [
    col
    for col in panel_cee_final.columns
    if col not in ["iso3", "country", "year"]
]

fully_missing_mask = (
    panel_cee_final[value_columns]
    .isna()
    .all(axis=1)
)

fully_missing_rows = (
    panel_cee_final.loc[
        fully_missing_mask,
        ["iso3", "country", "year"],
    ]
    .sort_values(["iso3", "year"])
)

print(
    "Liczba obserwacji państwo–rok bez dostępnych danych:",
    len(fully_missing_rows),
)

if not fully_missing_rows.empty:
    display(fully_missing_rows)

Liczba obserwacji państwo–rok bez dostępnych danych: 0


## 3. Konstrukcja zmiennych do analizy

Na podstawie połączonych danych źródłowych tworzone są zmienne wykorzystywane w analizie empirycznej. Realny PKB oraz nakłady brutto są przeliczane na pracownika i logarytmowane. Roczny wzrost gospodarczy jest przybliżany pierwszą różnicą logarytmu PKB na pracownika.

In [35]:
economic_numeric_columns = [
    "gdp_const",
    "labor_force",
    "gcf_const",
    "inflation_cpi",
    "fdi_gdp",
    "gov_consumption_const",
    "broad_money_gdp",
    "education_years_adults",
]

for col in economic_numeric_columns:
    panel_cee_final[col] = pd.to_numeric(
        panel_cee_final[col],
        errors="coerce",
    )


valid_labor = (
    panel_cee_final["labor_force"].notna()
    & (panel_cee_final["labor_force"] > 0)
)

panel_cee_final["gdp_per_worker"] = (
    panel_cee_final["gdp_const"]
    / panel_cee_final["labor_force"]
).where(valid_labor)

panel_cee_final["gcf_per_worker"] = (
    panel_cee_final["gcf_const"]
    / panel_cee_final["labor_force"]
).where(valid_labor)


panel_cee_final["ln_gdp_per_worker"] = np.log(
    panel_cee_final["gdp_per_worker"].where(
        panel_cee_final["gdp_per_worker"] > 0
    )
)

panel_cee_final["ln_gcf_per_worker"] = np.log(
    panel_cee_final["gcf_per_worker"].where(
        panel_cee_final["gcf_per_worker"] > 0
    )
)


panel_cee_final["d_ln_gdp_per_worker"] = (
    panel_cee_final
    .groupby("iso3")["ln_gdp_per_worker"]
    .diff()
)

panel_cee_final["gdp_per_worker_growth_pct"] = (
    panel_cee_final["d_ln_gdp_per_worker"]
    * 100
)


created_variables = [
    "gdp_per_worker",
    "gcf_per_worker",
    "ln_gdp_per_worker",
    "ln_gcf_per_worker",
    "d_ln_gdp_per_worker",
]

print(
    "Liczba obserwacji z niepoprawnym "
    "lub brakującym mianownikiem:",
    int((~valid_labor).sum()),
)

created_variables_missingness = (
    panel_cee_final[
        created_variables
    ]
    .isna()
    .sum()
    .rename("liczba_brakow")
    .rename_axis("zmienna")
    .to_frame()
)

show_table(
    created_variables_missingness,
    index=True,
)

Liczba obserwacji z niepoprawnym lub brakującym mianownikiem: 0


,liczba_brakow
zmienna,
gdp_per_worker,0
gcf_per_worker,26
ln_gdp_per_worker,0
ln_gcf_per_worker,26
d_ln_gdp_per_worker,10


### 3.1. Zmienne okresów kryzysowych

Utworzono cztery zmienne binarne identyfikujące okresy kryzysowe wykorzystane później w analizie interakcji z indeksem globalizacji.

In [ ]:
panel_cee_final["crisis_financial_2008_2009"] = (
    panel_cee_final["year"]
    .between(2008, 2009)
    .astype(int)
)

panel_cee_final["crisis_debt_2010_2012"] = (
    panel_cee_final["year"]
    .between(2010, 2012)
    .astype(int)
)

panel_cee_final["crisis_covid_2020_2021"] = (
    panel_cee_final["year"]
    .between(2020, 2021)
    .astype(int)
)

panel_cee_final["crisis_energy_war_2022_2023"] = (
    panel_cee_final["year"]
    .between(2022, 2023)
    .astype(int)
)


crisis_columns = [
    "crisis_financial_2008_2009",
    "crisis_debt_2010_2012",
    "crisis_covid_2020_2021",
    "crisis_energy_war_2022_2023",
]

crisis_year_check = (
    panel_cee_final[
        ["year"] + crisis_columns
    ]
    .drop_duplicates()
    .sort_values("year")
)


crisis_periods = pd.DataFrame({
    "zmienna": crisis_columns,
    "lata": [
        crisis_year_check.loc[
            crisis_year_check[col] == 1,
            "year",
        ].tolist()
        for col in crisis_columns
    ],
})

show_table(crisis_periods)

,zmienna,lata
0,crisis_financial_2008_2009,"[2008, 2009]"
1,crisis_debt_2010_2012,"[2010, 2011, 2012]"
2,crisis_covid_2020_2021,"[2020, 2021]"
3,crisis_energy_war_2022_2023,"[2022, 2023]"


## 4. Dostępność danych i wybór próby badawczej

Przed ustaleniem finalnego okresu analizy sprawdzono dostępność podstawowych zmiennych w skonstruowanym panelu. Pozwala to oddzielić braki wynikające z danych źródłowych od braków powstających wskutek transformacji zmiennych.

In [19]:
kof_columns_final = [
    col
    for col in panel_cee_final.columns
    if col.startswith("kof_")
]

key_columns = [
    "gdp_const",
    "labor_force",
    "gcf_const",
    "ln_gdp_per_worker",
    "ln_gcf_per_worker",
    "d_ln_gdp_per_worker",
    "inflation_cpi",
    "fdi_gdp",
    "gov_consumption_const",
    "broad_money_gdp",
    "education_years_adults",
] + kof_columns_final

coverage_records = []

for col in key_columns:
    non_missing = panel_cee_final[col].notna()

    available_years = panel_cee_final.loc[
        non_missing,
        "year",
    ]

    coverage_records.append({
        "zmienna": col,
        "wszystkie_obserwacje": len(
            panel_cee_final
        ),
        "niebrakujace_obserwacje": int(
            non_missing.sum()
        ),
        "liczba_brakow": int(
            panel_cee_final[col].isna().sum()
        ),
        "braki_proc": round(
            panel_cee_final[col].isna().mean()
            * 100,
            2,
        ),
        "pierwszy_rok": (
            int(available_years.min())
            if not available_years.empty
            else np.nan
        ),
        "ostatni_rok": (
            int(available_years.max())
            if not available_years.empty
            else np.nan
        ),
    })

coverage_table = pd.DataFrame(
    coverage_records
)

show_table(coverage_table)

zmienna,wszystkie_obserwacje,niebrakujace_obserwacje,liczba_brakow,braki_proc,pierwszy_rok,ostatni_rok
gdp_const,340,340,0,0.00,1990,2023
labor_force,340,340,0,0.00,1990,2023
gcf_const,340,314,26,7.65,1990,2023
ln_gdp_per_worker,340,340,0,0.00,1990,2023
ln_gcf_per_worker,340,314,26,7.65,1990,2023
d_ln_gdp_per_worker,340,330,10,2.94,1991,2023
inflation_cpi,340,328,12,3.53,1990,2023
fdi_gdp,340,319,21,6.18,1990,2023
gov_consumption_const,340,319,21,6.18,1990,2023
broad_money_gdp,340,166,174,51.18,1990,2023


### 4.1. Dostępność danych w latach 1990–1994

Ponieważ dane źródłowe obejmują również lata poprzedzające finalny okres badania, sprawdzono kompletność podstawowych zmiennych w latach 1990–1994. Analiza ta służy uzasadnieniu wyboru roku 1995 jako początku próby wykorzystywanej w dalszej analizie.

In [20]:
early_period = panel_cee_final.loc[
    panel_cee_final["year"].between(
        1990,
        1994,
    )
].copy()

main_model_variables = [
    "ln_gdp_per_worker",
    "ln_gcf_per_worker",
    "kof_overall",
    "education_years_adults",
    "inflation_cpi",
    "fdi_gdp",
    "gov_consumption_const",
]

early_coverage_records = []

for col in main_model_variables:
    early_coverage_records.append({
        "zmienna": col,
        "wszystkie_obserwacje_1990_1994": len(
            early_period
        ),
        "niebrakujace_obserwacje": int(
            early_period[col].notna().sum()
        ),
        "liczba_brakow": int(
            early_period[col].isna().sum()
        ),
        "pokrycie_proc": round(
            early_period[col].notna().mean()
            * 100,
            2,
        ),
    })

early_coverage_table = pd.DataFrame(
    early_coverage_records
)

show_table(early_coverage_table)

early_complete_cases = (
    early_period[
        main_model_variables
    ]
    .notna()
    .all(axis=1)
)

print(
    "Liczba kompletnych obserwacji "
    "dla podstawowego zestawu zmiennych "
    "w latach 1990–1994:",
    int(early_complete_cases.sum()),
    "z",
    len(early_period),
)

early_complete_by_country = (
    early_period
    .assign(
        complete_case=early_complete_cases
    )
    .groupby(["iso3", "country"])
    .agg(
        wszystkie_lata=("year", "size"),
        kompletne_lata=(
            "complete_case",
            "sum",
        ),
    )
    .reset_index()
)

early_complete_by_country["pokrycie_proc"] = (
    early_complete_by_country["kompletne_lata"]
    / early_complete_by_country["wszystkie_lata"]
    * 100
).round(2)

show_table(early_complete_by_country)

zmienna,wszystkie_obserwacje_1990_1994,niebrakujace_obserwacje,liczba_brakow,pokrycie_proc
ln_gdp_per_worker,50,50,0,100.0
ln_gcf_per_worker,50,24,26,48.0
kof_overall,50,42,8,84.0
education_years_adults,50,50,0,100.0
inflation_cpi,50,38,12,76.0
fdi_gdp,50,29,21,58.0
gov_consumption_const,50,29,21,58.0


Liczba kompletnych obserwacji dla podstawowego zestawu zmiennych w latach 1990–1994: 17 z 50


iso3,country,wszystkie_lata,kompletne_lata,pokrycie_proc
BGR,Bulgaria,5,0,0.0
CZE,Czechia,5,2,40.0
EST,Estonia,5,2,40.0
HUN,Hungary,5,4,80.0
LTU,Lithuania,5,0,0.0
LVA,Latvia,5,0,0.0
POL,Poland,5,0,0.0
ROU,Romania,5,4,80.0
SVK,Slovakia,5,2,40.0
SVN,Slovenia,5,3,60.0


In [21]:
EXPECTED_YEARS = (
    ANALYSIS_END_YEAR
    - ANALYSIS_START_YEAR
    + 1
)

EXPECTED_OBSERVATIONS = (
    len(CEE) * EXPECTED_YEARS
)

assert len(panel_cee_final) == EXPECTED_OBSERVATIONS, (
    "Nieprawidłowa liczba wierszy. "
    f"Oczekiwano {EXPECTED_OBSERVATIONS}, "
    f"otrzymano {len(panel_cee_final)}."
)

assert (
    panel_cee_final["iso3"].nunique()
    == len(CEE)
), (
    "Nieprawidłowa liczba państw. "
    f"Oczekiwano {len(CEE)}."
)

assert (
    panel_cee_final.duplicated(
        ["iso3", "year"]
    ).sum()
    == 0
), "W finalnym panelu pozostały duplikaty."

assert (
    country_year_check["liczba_lat"]
    .eq(EXPECTED_YEARS)
    .all()
), (
    "Nie każde państwo ma oczekiwaną "
    f"liczbę {EXPECTED_YEARS} lat."
)

assert (
    country_year_check["pierwszy_rok"]
    .eq(ANALYSIS_START_YEAR)
    .all()
), (
    "Nie każde państwo zaczyna się "
    f"w {ANALYSIS_START_YEAR} roku."
)

assert (
    country_year_check["ostatni_rok"]
    .eq(ANALYSIS_END_YEAR)
    .all()
), (
    "Nie każde państwo kończy się "
    f"w {ANALYSIS_END_YEAR} roku."
)

print(
    f"Panel diagnostyczny obejmuje "
    f"{len(CEE)} państw, "
    f"{EXPECTED_YEARS} lat i "
    f"{EXPECTED_OBSERVATIONS} obserwacji państwo–rok."
)

Panel diagnostyczny obejmuje 10 państw, 34 lat i 340 obserwacji państwo–rok.


### 4.2. Porównanie alternatywnych okresów próby

Kompletność podstawowego zestawu zmiennych porównano dla pełnego dostępneg ookresu 1990–2023 oraz dla okresu 1995–2023 przyjętego następnie jako główna próba badawcza.

In [22]:
def compare_model_sample(
    df: pd.DataFrame,
    start_year: int,
    end_year: int,
    variables: list[str],
) -> dict:
    sample = df.loc[
        df["year"].between(
            start_year,
            end_year,
        )
    ].copy()

    complete = (
        sample[variables]
        .notna()
        .all(axis=1)
    )

    return {
        "okres": f"{start_year}–{end_year}",
        "wiersze_strukturalne": len(sample),
        "kompletne_obserwacje_modelowe": int(
            complete.sum()
        ),
        "udzial_kompletnych_proc": round(
            complete.mean() * 100,
            2,
        ),
        "liczba_panstw": sample.loc[
            complete,
            "iso3",
        ].nunique(),
    }


sample_comparison = pd.DataFrame([
    compare_model_sample(
        panel_cee_final,
        1990,
        2023,
        main_model_variables,
    ),
    compare_model_sample(
        panel_cee_final,
        1995,
        2023,
        main_model_variables,
    ),
])

show_table(sample_comparison)

okres,wiersze_strukturalne,kompletne_obserwacje_modelowe,udzial_kompletnych_proc,liczba_panstw
1990–2023,340,307,90.29,10
1995–2023,290,290,100.00,10


### 4.3. Struktura braków danych

Dla podstawowych zmiennych wykorzystywanych w analizie sprawdzono rozmieszczenie braków według państw i lat. Dodatkowo zweryfikowano, czy braki występują wyłącznie na początku lub końcu szeregów, czy również wewnątrz okresu dostępności danych.

In [23]:
main_missingness_variables = [
    "ln_gdp_per_worker",
    "ln_gcf_per_worker",
    "kof_overall",
    "education_years_adults",
    "inflation_cpi",
    "fdi_gdp",
    "gov_consumption_const",
]

In [24]:
missing_by_country = (
    panel_cee_final
    .groupby(["iso3", "country"])[
        main_missingness_variables
    ]
    .agg(lambda series: int(series.isna().sum()))
    .reset_index()
)

missing_by_country_long = (
    missing_by_country
    .melt(
        id_vars=["iso3", "country"],
        var_name="zmienna",
        value_name="liczba_brakow",
    )
    .query("liczba_brakow > 0")
    .sort_values(
        ["iso3", "liczba_brakow", "zmienna"],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

show_table(missing_by_country_long)

iso3,country,zmienna,liczba_brakow
BGR,Bulgaria,ln_gcf_per_worker,5
CZE,Czechia,fdi_gdp,3
CZE,Czechia,kof_overall,3
CZE,Czechia,inflation_cpi,2
EST,Estonia,fdi_gdp,3
EST,Estonia,gov_consumption_const,3
EST,Estonia,inflation_cpi,3
EST,Estonia,ln_gcf_per_worker,3
EST,Estonia,kof_overall,1
HUN,Hungary,gov_consumption_const,1


In [25]:
missing_years_records = []

for col in main_missingness_variables:
    for iso3, country_df in panel_cee_final.groupby("iso3"):
        missing_years = (
            country_df.loc[
                country_df[col].isna(),
                "year",
            ]
            .astype(int)
            .tolist()
        )

        if missing_years:
            missing_years_records.append({
                "iso3": iso3,
                "country": country_df["country"].iloc[0],
                "zmienna": col,
                "liczba_brakow": len(missing_years),
                "brakujace_lata": missing_years,
            })

missing_years_table = (
    pd.DataFrame(missing_years_records)
    .sort_values(
        ["iso3", "zmienna"]
    )
    .reset_index(drop=True)
)

show_table(missing_years_table)

iso3,country,zmienna,liczba_brakow,brakujace_lata
BGR,Bulgaria,ln_gcf_per_worker,5,"[1990, 1991, 1992, 1993, 1994]"
CZE,Czechia,fdi_gdp,3,"[1990, 1991, 1992]"
CZE,Czechia,inflation_cpi,2,"[1990, 1991]"
CZE,Czechia,kof_overall,3,"[1990, 1991, 1992]"
EST,Estonia,fdi_gdp,3,"[1990, 1991, 1992]"
EST,Estonia,gov_consumption_const,3,"[1990, 1991, 1992]"
EST,Estonia,inflation_cpi,3,"[1990, 1991, 1992]"
EST,Estonia,kof_overall,1,[1990]
EST,Estonia,ln_gcf_per_worker,3,"[1990, 1991, 1992]"
HUN,Hungary,gov_consumption_const,1,[1990]


In [26]:
year_completeness = (
    panel_cee_final
    .assign(
        complete_main_model=(
            panel_cee_final[
                main_missingness_variables
            ]
            .notna()
            .all(axis=1)
        )
    )
    .groupby("year")
    .agg(
        liczba_panstw=("iso3", "nunique"),
        kompletne_panstwa=(
            "complete_main_model",
            "sum",
        ),
    )
    .reset_index()
)

year_completeness["niekompletne_panstwa"] = (
    year_completeness["liczba_panstw"]
    - year_completeness["kompletne_panstwa"]
)

year_completeness["pokrycie_proc"] = (
    year_completeness["kompletne_panstwa"]
    / year_completeness["liczba_panstw"]
    * 100
).round(2)

show_table(
    year_completeness.loc[
        year_completeness["pokrycie_proc"] < 100
    ]
)

year,liczba_panstw,kompletne_panstwa,niekompletne_panstwa,pokrycie_proc
1990,10,0,10,0.0
1991,10,2,8,20.0
1992,10,3,7,30.0
1993,10,6,4,60.0
1994,10,6,4,60.0


In [27]:
internal_gap_records = []

for col in main_missingness_variables:

    for iso3, country_df in panel_cee_final.groupby("iso3"):

        country_df = (
            country_df
            .sort_values("year")
            .copy()
        )

        available_years = (
            country_df.loc[
                country_df[col].notna(),
                "year",
            ]
            .astype(int)
        )

        if available_years.empty:
            continue

        first_available = available_years.min()
        last_available = available_years.max()

        internal_missing_years = (
            country_df.loc[
                country_df["year"].between(
                    first_available,
                    last_available,
                )
                & country_df[col].isna(),
                "year",
            ]
            .astype(int)
            .tolist()
        )

        if internal_missing_years:
            internal_gap_records.append({
                "iso3": iso3,
                "country": country_df["country"].iloc[0],
                "zmienna": col,
                "brakujace_lata_wewnetrzne":
                    internal_missing_years,
            })


internal_gap_cases = pd.DataFrame(
    internal_gap_records
)

print(
    "Liczba szeregów z lukami wewnętrznymi:",
    len(internal_gap_cases),
)

if not internal_gap_cases.empty:
    display(internal_gap_cases)

Liczba szeregów z lukami wewnętrznymi: 0


### 4.4. Dostępność zmiennej broad money

Zmienna `broad_money_gdp` nie należy do podstawowego zestawu zmiennych
wykorzystywanego przy wyborze głównej próby. Ze względu na ograniczoną
dostępność sprawdzono jej pokrycie oddzielnie dla poszczególnych państw.

In [28]:
broad_money_records = []

for iso3, country_df in panel_cee_final.groupby("iso3"):

    available_years = (
        country_df.loc[
            country_df["broad_money_gdp"].notna(),
            "year",
        ]
        .astype(int)
        .tolist()
    )

    broad_money_records.append({
        "iso3":
            iso3,

        "country":
            country_df["country"].iloc[0],

        "liczba_dostepnych_lat":
            len(available_years),

        "pierwszy_dostepny_rok":
            (
                min(available_years)
                if available_years
                else np.nan
            ),

        "ostatni_dostepny_rok":
            (
                max(available_years)
                if available_years
                else np.nan
            ),
    })


broad_money_diagnostics = pd.DataFrame(
    broad_money_records
)

show_table(broad_money_diagnostics)

iso3,country,liczba_dostepnych_lat,pierwszy_dostepny_rok,ostatni_dostepny_rok
BGR,Bulgaria,33,1991.0,2023.0
CZE,Czechia,31,1993.0,2023.0
EST,Estonia,0,NaN,NaN
HUN,Hungary,34,1990.0,2023.0
LTU,Lithuania,0,NaN,NaN
LVA,Latvia,0,NaN,NaN
POL,Poland,34,1990.0,2023.0
ROU,Romania,34,1990.0,2023.0
SVK,Slovakia,0,NaN,NaN
SVN,Slovenia,0,NaN,NaN


## 5. Finalna próba i eksport danych

Na podstawie analizy dostępności danych finalną próbę ograniczono do lat
1995–2023. Poniżej tworzony jest panel wykorzystywany w dalszej analizie
empirycznej oraz wykonywane są końcowe kontrole jego struktury.

In [29]:
EXPORT_START_YEAR = 1995
EXPORT_END_YEAR = 2023

panel_export = (
    panel_cee_final.loc[
        panel_cee_final["year"].between(
            EXPORT_START_YEAR,
            EXPORT_END_YEAR,
        )
    ]
    .copy()
    .sort_values(["iso3", "year"])
    .reset_index(drop=True)
)

expected_export_years = (
    EXPORT_END_YEAR
    - EXPORT_START_YEAR
    + 1
)

expected_export_observations = (
    len(CEE) * expected_export_years
)

print(
    "Finalna próba:",
    f"{EXPORT_START_YEAR}–{EXPORT_END_YEAR}",
)

print(
    "Liczba obserwacji:",
    len(panel_export),
)

print(
    "Liczba państw:",
    panel_export["iso3"].nunique(),
)

Finalna próba: 1995–2023
Liczba obserwacji: 290
Liczba państw: 10


In [30]:
assert len(panel_export) == expected_export_observations, (
    "Nieprawidłowa liczba obserwacji w próbie eksportowej. "
    f"Oczekiwano {expected_export_observations}, "
    f"otrzymano {len(panel_export)}."
)

assert (
    panel_export["iso3"].nunique()
    == len(CEE)
), "Nieprawidłowa liczba państw."

assert (
    panel_export.duplicated(
        subset=["iso3", "year"]
    ).sum()
    == 0
), "W próbie eksportowej występują duplikaty."

assert panel_export["year"].min() == EXPORT_START_YEAR
assert panel_export["year"].max() == EXPORT_END_YEAR

In [31]:
country_year_check_export = (
    panel_export
    .groupby(["iso3", "country"])
    .agg(
        liczba_lat=("year", "nunique"),
        pierwszy_rok=("year", "min"),
        ostatni_rok=("year", "max"),
    )
    .reset_index()
)

show_table(country_year_check_export)

iso3,country,liczba_lat,pierwszy_rok,ostatni_rok
BGR,Bulgaria,29,1995,2023
CZE,Czechia,29,1995,2023
EST,Estonia,29,1995,2023
HUN,Hungary,29,1995,2023
LTU,Lithuania,29,1995,2023
LVA,Latvia,29,1995,2023
POL,Poland,29,1995,2023
ROU,Romania,29,1995,2023
SVK,Slovakia,29,1995,2023
SVN,Slovenia,29,1995,2023


In [32]:
kof_columns_export = [
    col
    for col in panel_export.columns
    if col.startswith("kof_")
]

coverage_variables_export = [
    "gdp_const",
    "labor_force",
    "gcf_const",
    "gdp_per_worker",
    "gcf_per_worker",
    "ln_gdp_per_worker",
    "ln_gcf_per_worker",
    "d_ln_gdp_per_worker",
    "inflation_cpi",
    "fdi_gdp",
    "gov_consumption_const",
    "broad_money_gdp",
    "education_years_adults",
] + kof_columns_export

coverage_export_records = []

for col in coverage_variables_export:
    non_missing = panel_export[col].notna()

    available_years = panel_export.loc[
        non_missing,
        "year",
    ]

    coverage_export_records.append({
        "zmienna": col,
        "wszystkie_obserwacje": len(panel_export),
        "niebrakujace_obserwacje": int(
            non_missing.sum()
        ),
        "liczba_brakow": int(
            panel_export[col].isna().sum()
        ),
        "braki_proc": round(
            panel_export[col].isna().mean() * 100,
            2,
        ),
        "pierwszy_rok": (
            int(available_years.min())
            if not available_years.empty
            else np.nan
        ),
        "ostatni_rok": (
            int(available_years.max())
            if not available_years.empty
            else np.nan
        ),
    })

coverage_table_export = pd.DataFrame(
    coverage_export_records
)

show_table(coverage_table_export)

zmienna,wszystkie_obserwacje,niebrakujace_obserwacje,liczba_brakow,braki_proc,pierwszy_rok,ostatni_rok
gdp_const,290,290,0,0.0,1995,2023
labor_force,290,290,0,0.0,1995,2023
gcf_const,290,290,0,0.0,1995,2023
gdp_per_worker,290,290,0,0.0,1995,2023
gcf_per_worker,290,290,0,0.0,1995,2023
ln_gdp_per_worker,290,290,0,0.0,1995,2023
ln_gcf_per_worker,290,290,0,0.0,1995,2023
d_ln_gdp_per_worker,290,290,0,0.0,1995,2023
inflation_cpi,290,290,0,0.0,1995,2023
fdi_gdp,290,290,0,0.0,1995,2023


### 5.1. Eksport finalnego panelu

Finalny panel oraz tabele kontrolne są zapisywane do katalogu `output`.

In [33]:
panel_csv_path = (
    OUTPUT_DIR
    / "panel_cee_1995_2023.csv"
)

panel_excel_path = (
    OUTPUT_DIR
    / "panel_cee_1995_2023.xlsx"
)

coverage_path = (
    OUTPUT_DIR
    / "panel_cee_coverage_1995_2023.xlsx"
)

country_year_path = (
    OUTPUT_DIR
    / "panel_cee_country_year_check_1995_2023.xlsx"
)

panel_export.to_csv(
    panel_csv_path,
    index=False,
    encoding="utf-8-sig",
)

panel_export.to_excel(
    panel_excel_path,
    index=False,
)

coverage_table_export.to_excel(
    coverage_path,
    index=False,
)

country_year_check_export.to_excel(
    country_year_path,
    index=False,
)

print("Zapisano finalne pliki:")
print(f"1. {panel_csv_path}")
print(f"2. {panel_excel_path}")
print(f"3. {coverage_path}")
print(f"4. {country_year_path}")

Zapisano finalne pliki:
1. ..\output\panel_cee_1995_2023.csv
2. ..\output\panel_cee_1995_2023.xlsx
3. ..\output\panel_cee_coverage_1995_2023.xlsx
4. ..\output\panel_cee_country_year_check_1995_2023.xlsx


In [34]:
export_check = pd.read_csv(
    panel_csv_path
)

print("KONTROLA WYEKSPORTOWANEGO PLIKU")
print("Liczba wierszy:", len(export_check))
print("Liczba kolumn:", export_check.shape[1])
print(
    "Liczba państw:",
    export_check["iso3"].nunique(),
)
print(
    "Zakres lat:",
    export_check["year"].min(),
    "-",
    export_check["year"].max(),
)
print(
    "Duplikaty iso3–year:",
    export_check.duplicated(
        subset=["iso3", "year"]
    ).sum(),
)

assert len(export_check) == 290
assert export_check["iso3"].nunique() == 10
assert export_check["year"].min() == 1995
assert export_check["year"].max() == 2023

assert (
    export_check.duplicated(
        subset=["iso3", "year"]
    ).sum()
    == 0
)

print("Eksport 1995–2023 zakończony poprawnie.")

KONTROLA WYEKSPORTOWANEGO PLIKU
Liczba wierszy: 290
Liczba kolumn: 48
Liczba państw: 10
Zakres lat: 1995 - 2023
Duplikaty iso3–year: 0
Eksport 1995–2023 zakończony poprawnie.
